R packages: 
Rscript -e 'install.packages(c("brms", "posterior", "dplyr", "tidyr", "readr"), repos="https://cloud.r-project.org")'

In [7]:
# Feasibility Analysis for RQ2 - inspection of zero-shot responses
import pandas as pd
from pathlib import Path
df_zeroshot = pd.read_csv('../1_zero-shot/zeroshotResponses/zeroshotFeasibilityResponses.csv')

df_zeroshot.info()
unique_zeroshot = df_zeroshot['iteration'].nunique()
print(f"Unique iteration in zero-shot responses: {unique_zeroshot}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   row_id        600 non-null    object
 1   variant_id    600 non-null    object
 2   base_model    600 non-null    object
 3   model         600 non-null    object
 4   rating        600 non-null    int64 
 5   label         600 non-null    object
 6   iteration     600 non-null    int64 
 7   timestamp     600 non-null    object
 8   raw_response  600 non-null    object
dtypes: int64(2), object(7)
memory usage: 42.3+ KB
Unique iteration in zero-shot responses: 50


In [2]:
#
import pandas as pd

df_llm = pd.read_csv('../1_zero-shot/zeroshotResponses/zeroshotFeasibilityResponses.csv')

# inspect column names first
print(df_llm.columns)

# filter rows for mistral
mistral_rows = df_llm[df_llm['base_model'].str.contains('gemma3', case=False, na=False)]

# show the ratings provided by mistral
print(mistral_rows[['base_model', 'rating']])

# if you only want the unique ratings
print(mistral_rows['rating'].unique())

Index(['row_id', 'variant_id', 'base_model', 'model', 'rating', 'label',
       'iteration', 'timestamp', 'raw_response'],
      dtype='object')
    base_model  rating
3       gemma3       3
4       gemma3       3
5       gemma3       3
15      gemma3       3
16      gemma3       3
..         ...     ...
580     gemma3       3
581     gemma3       3
591     gemma3       3
592     gemma3       3
593     gemma3       3

[150 rows x 2 columns]
[3]


In [8]:
# Feasibility Analysis for RQ2 - inspection of context responses
import pandas as pd
from pathlib import Path
df_ctx = pd.read_csv('../2_context/contextResponses/contextFeasibilityResponses.csv')

df_ctx.info()

unique_ctx = df_ctx['iteration'].nunique()
print(f"Unique iteration in context responses: {unique_ctx}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   row_id        600 non-null    object
 1   variant_id    600 non-null    object
 2   base_model    600 non-null    object
 3   model         600 non-null    object
 4   rating        600 non-null    int64 
 5   label         600 non-null    object
 6   iteration     600 non-null    int64 
 7   timestamp     600 non-null    object
 8   raw_response  600 non-null    object
dtypes: int64(2), object(7)
memory usage: 42.3+ KB
Unique iteration in context responses: 50


In [10]:
# cell3: merging zeroshot and context data of feasibility responses
import pandas as pd
# create output directory
output_dir = Path('feas-ctx-zeroshot-working')
output_dir.mkdir(parents=True, exist_ok=True)

# load data
df_feas_ctx = pd.read_csv('../2_context/contextResponses/contextFeasibilityResponses.csv')
df_feas_zeroshot = pd.read_csv('../1_zero-shot/zeroshotResponses/zeroshotFeasibilityResponses.csv')

# define source and condition
df_feas_ctx['condition'] = 'context'
df_feas_zeroshot['condition'] = 'zeroshot'

# exact columns to keep
columns_to_keep = ['row_id', 'variant_id', 'base_model', 'model', 'rating', 'label', 'condition', 'iteration']

# combine the dataframes
df_combined = pd.concat([df_feas_ctx[columns_to_keep], df_feas_zeroshot[columns_to_keep]], ignore_index=True)

# save combined dataframe
df_combined.to_csv(output_dir / 'feas-ctx-zeroshot-responses.csv', index=False)

df_combined.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   row_id      1200 non-null   object
 1   variant_id  1200 non-null   object
 2   base_model  1200 non-null   object
 3   model       1200 non-null   object
 4   rating      1200 non-null   int64 
 5   label       1200 non-null   object
 6   condition   1200 non-null   object
 7   iteration   1200 non-null   int64 
dtypes: int64(2), object(6)
memory usage: 75.1+ KB


In [ ]:
# latext table for Bayesian model fit
from pathlib import Path
import pandas as pd
from IPython.display import display

working_dir = Path("feas-ctx-zeroshot-working")

input_path = (
    working_dir
    / "bayesian-results"
    / "prior-sensitivity"
    / "rq2_feasibility_prior_sensitivity_effects.csv"
)

output_path = (
    working_dir
    / "rq2_feasibility_primary_results_table.tex"
)

if not input_path.exists():
    raise FileNotFoundError(
        f"RQ2 sensitivity results not found:\n{input_path.resolve()}"
    )

model_labels = {
    "gemma3": "Gemma3:12B",
    "llama": "LLaMa-Pro",
    "mistral": "Mistral",
    "phi4": "Phi-4",
}

model_order = {
    "gemma3": 1,
    "llama": 2,
    "mistral": 3,
    "phi4": 4,
}

conclusion_labels = {
    "higher in Context": "Higher in Context",
    "lower in Context": "Lower in Context",
    "uncertain": "Uncertain",
}


def format_probability(value):
    """Avoid displaying a posterior draw proportion as absolute certainty."""
    value = float(value)

    if value >= 0.9995:
        return r"$>0.999$"
    if value <= 0.0005:
        return r"$<0.001$"

    return f"{value:.3f}"


# Read the complete prior-sensitivity results
results = pd.read_csv(input_path)

required_columns = {
    "prior_name",
    "base_model",
    "OR_median",
    "OR_l95",
    "OR_u95",
    "posterior_probability_OR_gt_1",
    "conclusion",
}

missing_columns = required_columns.difference(results.columns)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

# The primary-prior estimates are used for the main results table
primary = results.loc[
    results["prior_name"].eq("primary")
].copy()

if len(primary) != 4:
    raise ValueError(
        f"Expected 4 RQ2 primary rows, found {len(primary)}."
    )

if set(primary["base_model"]) != set(model_labels):
    raise ValueError(
        "The RQ2 table does not contain the expected four base models."
    )

primary["model_order"] = primary["base_model"].map(model_order)
primary = (
    primary
    .sort_values("model_order")
    .reset_index(drop=True)
)

table = pd.DataFrame({
    "Model": primary["base_model"].map(model_labels),
    "OR": primary["OR_median"].map(
        lambda value: f"{value:.3f}"
    ),
    r"95\% CrI": [
        f"[{lower:.3f}, {upper:.3f}]"
        for lower, upper in zip(
            primary["OR_l95"],
            primary["OR_u95"],
        )
    ],
    r"$P(OR>1)$": primary[
        "posterior_probability_OR_gt_1"
    ].map(format_probability),
    "Interpretation": primary["conclusion"].map(
        conclusion_labels
    ),
})

latex = table.to_latex(
    index=False,
    escape=False,
    column_format="lcccc",
    caption=(
        "RQ2 feasibility: posterior effects of Context relative "
        "to Zero-shot under the primary prior."
    ),
    label="tab:rq2_feasibility_bayesian",
    position="tbp",
)

note = (
    "\\end{tabular}\n"
    "\\vspace{2pt}\n"
    "\\begin{minipage}{\\linewidth}\n"
    "\\footnotesize\\textit{Note.} "
    "OR $>1$ indicates higher odds of receiving a higher "
    "feasibility rating in Context relative to Zero-shot. "
    "CrI denotes the 95\\% Bayesian credible interval. "
    "The primary effect prior was Normal$(0,1.5)$.\n"
    "\\end{minipage}"
)

latex = latex.replace(
    "\\end{tabular}",
    note,
    1,
)

output_path.write_text(latex, encoding="utf-8")

print(f"RQ2 LaTeX table saved to:\n{output_path.resolve()}")
display(table)

RQ2 LaTeX table saved to:
/Users/HP/Documents/second-publication/current-analysis/re-run/rq2-rerun/feas-ctx-zeroshot-working/rq2_feasibility_primary_results_table.tex


,Model,OR,95\% CrI,$P(OR>1)$,Interpretation
0,Gemma3:12B,1.502,"[0.292, 8.297]",0.687,Uncertain
1,LLaMa-Pro,3.254,"[2.026, 5.337]",$>0.999$,Higher in Context
2,Mistral,1.504,"[0.306, 8.542]",0.691,Uncertain
3,Phi-4,0.139,"[0.047, 0.346]",$<0.001$,Lower in Context


In [3]:
# sensitivity results comparisons: 
from pathlib import Path
import pandas as pd
from IPython.display import display

working_dir = Path("feas-ctx-zeroshot-working")

input_path = (
    working_dir
    / "bayesian-results"
    / "prior-sensitivity"
    / "rq2_feasibility_prior_sensitivity_effects.csv"
)

output_path = (
    working_dir
    / "rq2_feasibility_prior_sensitivity_table.tex"
)

if not input_path.exists():
    raise FileNotFoundError(
        f"Sensitivity results not found:\n{input_path.resolve()}"
    )

model_labels = {
    "gemma3": "Gemma3:12B",
    "llama": "LLaMa-Pro",
    "mistral": "Mistral",
    "phi4": "Phi-4",
}

model_order = [
    "phi4",
    "gemma3",
    "mistral",
    "llama",
]

prior_order = [
    "regularizing",
    "primary",
    "weak",
]

prior_labels = {
    "regularizing": "Regularizing",
    "primary": "Primary",
    "weak": "Weak",
}

conclusion_labels = {
    "higher in Context": "Higher in Context",
    "lower in Context": "Lower in Context",
    "uncertain": "Uncertain",
}


def format_number(value):
    value = float(value)

    if value != 0 and (
        abs(value) < 0.001 or abs(value) >= 1000
    ):
        mantissa, exponent = f"{value:.2e}".split("e")
        return (
            rf"${mantissa}\times 10^{{{int(exponent)}}}$"
        )

    return f"{value:.3f}"


def format_effect(row):
    return (
        f"{format_number(row['OR_median'])} "
        f"[{format_number(row['OR_l95'])}, "
        f"{format_number(row['OR_u95'])}]"
    )


results = pd.read_csv(input_path)

required_columns = {
    "prior_name",
    "base_model",
    "OR_median",
    "OR_l95",
    "OR_u95",
    "conclusion",
}

missing_columns = required_columns.difference(
    results.columns
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

if len(results) != 12:
    raise ValueError(
        "Expected 12 rows: 4 models × 3 priors; "
        f"found {len(results)}."
    )

records = []

for base_model in model_order:
    model_data = results.loc[
        results["base_model"].eq(base_model)
    ]

    if set(model_data["prior_name"]) != set(prior_order):
        raise ValueError(
            f"Incomplete prior results for {base_model}."
        )

    record = {
        "Model": model_labels[base_model],
    }

    conclusions = []

    for prior_name in prior_order:
        prior_row = model_data.loc[
            model_data["prior_name"].eq(prior_name)
        ].iloc[0]

        record[
            f"{prior_labels[prior_name]} OR (95\\% CrI)"
        ] = format_effect(prior_row)

        conclusions.append(
            conclusion_labels[prior_row["conclusion"]]
        )

    record["Conclusions (R/P/W)"] = " / ".join(
        conclusions
    )

    record["Stable"] = (
        "Yes"
        if len(set(conclusions)) == 1
        else "No"
    )

    records.append(record)

table = pd.DataFrame(records)

latex = table.to_latex(
    index=False,
    escape=False,
    column_format=(
        r"p{2.2cm}"
        r"p{3.3cm}"
        r"p{3.3cm}"
        r"p{3.3cm}"
        r"p{4.0cm}"
        r"c"
    ),
    caption=(
        "Prior-sensitivity comparison for the RQ2 feasibility "
        "effects of Context relative to Zero-shot."
    ),
    label="tab:rq2_feasibility_prior_sensitivity",
    position="tbp",
)

note = (
    "\\end{tabular}\n"
    "\\vspace{2pt}\n"
    "\\begin{minipage}{\\linewidth}\n"
    "\\footnotesize\\textit{Note.} "
    "Cells report posterior median odds ratios with 95\\% "
    "credible intervals. R/P/W denotes regularizing, primary, "
    "and weak priors, respectively. Stable indicates that the "
    "inferential conclusion was identical under all three priors. "
    "OR $>1$ indicates higher odds of a higher feasibility rating "
    "in Context relative to Zero-shot.\n"
    "\\end{minipage}"
)

latex = latex.replace(
    "\\end{tabular}",
    note,
    1,
)

output_path.write_text(latex, encoding="utf-8")

print(f"Saved to:\n{output_path.resolve()}")
display(table)

Saved to:
/Users/HP/Documents/second-publication/current-analysis/re-run/rq2-rerun/feas-ctx-zeroshot-working/rq2_feasibility_prior_sensitivity_table.tex


,Model,Regularizing OR (95\% CrI),Primary OR (95\% CrI),Weak OR (95\% CrI),Conclusions (R/P/W),Stable
0,Phi-4,"0.220 [0.097, 0.473]","0.139 [0.047, 0.346]","0.117 [0.037, 0.312]",Lower in Context / Lower in Context / Lower in...,Yes
1,Gemma3:12B,"1.232 [0.391, 3.908]","1.502 [0.292, 8.297]","1.684 [0.290, 11.686]",Uncertain / Uncertain / Uncertain,Yes
2,Mistral,"1.238 [0.384, 3.997]","1.504 [0.306, 8.542]","1.698 [0.275, 12.396]",Uncertain / Uncertain / Uncertain,Yes
3,LLaMa-Pro,"2.979 [1.883, 4.738]","3.254 [2.026, 5.337]","3.329 [2.050, 5.461]",Higher in Context / Higher in Context / Higher...,Yes
